In [1]:
import re
import sys
import os
import cv2
import numpy as np

from marsneuralzoo.models.e2emvm import E2emvm, GraphTracker

In [2]:
e2emvm = E2emvm(multiview=False)
# graph_tracker = GraphTracker(track_stride=3)

Loaded SuperPoint model


In [3]:
from dataengine.generator.obstacle.debug_utils import (
    draw_mask,
    visualize_optical_flow,
    load_cam_tracker_output,
    load_cam_params,
)
from dataengine.generator.obstacle.assignment import assign_segments

from dataengine.generator.obstacle.segment_tracker import (
    CamId,
    SegmentTracklet,
    SegmentTracker,
    TrackletDatabase,
)

from marsdataio.dbhelper import (
    collate_images,  # just for now
)

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars


In [48]:
from marsdataio.npyrenderer.renderer import generate_colors

colors = generate_colors()

In [65]:
import torch
from itertools import count
from lapsolver import solve_dense
import logging

if 'torch_images' in globals():
    del torch_images, features, preds
    torch.cuda.empty_cache()

# data_path = './cache/2023_04_18_09_29_56_seg_0'
# data_path = './cache/2023_04_28_15_35_01_seg_225'
# data_path = './cache/2023_04_27_17_07_32_seg_13'
# data_path = './cache/2023_07_27_07_32_43_seg_37'
# data_path = './cache/2023_11_02_18_22_28_seg_0' ### not good
# data_path = './cache/2023_02_27_15_21_49_seg_175_seq_3'
# data_path = './cache/2023_10_10_07_28_33_seg_12'
# data_path = './cache/2023_03_20_11_55_00_seg_4'
# data_path = './cache/2023_10_12_11_13_43_seg_8'
# data_path = './cache/2023_07_21_07_12_48_seg_3'
# data_path = './cache/2023_05_11_07_21_34_seg_0'
# data_path = './cache/2023_11_01_07_30_47_seg_241'
# data_path = './cache/2023_10_19_07_21_48_seg_53'
# data_path = './cache/2023_12_22_14_30_55_seg_197'
data_path = './cache/2023_10_19_08_34_35_seg_93'

data_base_name = os.path.basename(data_path)
cam_states_dict, tracklet_db = load_cam_tracker_output(data_path)

# print(data_base_name)
logging.getLogger().handlers.clear()
logging.basicConfig(
    filename=f"./cache/{data_base_name}/logfile.log",  # 로그 파일 이름
    level=logging.INFO,  # 로그 레벨 설정
    format="%(asctime)s - %(message)s",  # 로그 메시지 포맷 설정
    datefmt="%Y-%m-%d %H:%M:%S",  # 시간 형식 설정
    filemode="w",
)


cam_params = load_cam_params(data_path)


def skew(vector):
    x, y, z = vector
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])


target_pair = [
    # front
    (0, 1),
    (0, 2),
    # (1, 2),
    # left
    (2, 3),
    (3, 5),
    # right
    (2, 4),
    (4, 6),
]
threshold = 0.1
epipolar_threshold_list = [
    # front    # front
    0.1,  #    # (0, 1),
    0.001,  #   # (0, 2),
    # 0.001,  # (1, 2),
    # left     # left
    0.1,  #    # (2, 3),
    0.07,  #    # (3, 5),
    # right    # right
    0.1,  #    # (2, 4),
    0.1,  #    # (4, 6),
]
# min_match_count_list = [
#     # front    # front
#     10,  #    # (0, 1),
#     10,  #   # (0, 2),
#     # (1, 2),  # (1, 2),
#     # left     # left
#     10,  #    # (2, 3),
#     10,  #    # (3, 5),
#     # right    # right
#     10,  #    # (2, 4),
#     10,  #    # (4, 6),
# ]

target_Ec1c0 = {}
epipolar_threshold = {}
match_count_threshold = {}

match_video_frames = {}
for i, (idx0, idx1) in enumerate(target_pair):
    key = f'{idx0}_{idx1}'

    T_bc0 = cam_params[idx0].T_bc
    T_bc1 = cam_params[idx1].T_bc
    R_bc0 = T_bc0[:3, :3]
    P_bc0 = T_bc0[:3, 3]
    R_bc1 = T_bc1[:3, :3]
    P_bc1 = T_bc1[:3, 3]

    R_c1b = R_bc1.T
    P_c1b = -R_c1b @ P_bc1

    R_c1c0 = R_c1b @ R_bc0
    P_c1c0 = R_c1b @ P_bc0 + P_c1b
    P_skew = skew(P_c1c0)
    Ec1c0 = P_skew @ R_c1c0
    target_Ec1c0[key] = Ec1c0
    epipolar_threshold[key] = epipolar_threshold_list[i]
    # match_count_threshold[key] = min_match_count_list[i]

    match_video_frames[key] = []


skip = 0
# dura = 10
dura = 0
timestamp_count = 0
segment_id_pairs_dict = {}

for timestamp, cam_states in cam_states_dict.items():
    timestamp_count += 1
    if timestamp_count < skip:
        continue

    if skip + dura != 0 and timestamp_count > skip + dura:
        break
    print(f"processing {timestamp} {timestamp_count}")

    images = []
    for cam_state in cam_states:
        images.append(cam_state.image)

    torch_images, features = e2emvm.extract_features(images)
    preds = e2emvm(torch_images, features, target_pair)

    all_matches = preds['matches']

    kpt_seg_ids = []
    undistorted_kpts_list = []
    for cam_idx, kpts in enumerate(features['keypoints']):
        kpts = kpts.cpu().numpy().astype(np.float32)
        seg_ids = np.full(kpts.shape[0], -1, dtype=int)
        id_image = cam_states_dict[timestamp][cam_idx].draw_id()

        for idx, kpt in enumerate(kpts):
            x, y = kpt.astype(int)
            seg_id = id_image[y, x]
            if seg_id >= 0:
                seg_ids[idx] = seg_id

        kpt_seg_ids.append(seg_ids)

        K = cam_params[cam_idx].K
        D = cam_params[cam_idx].distort_coefs
        R = np.eye(3, dtype=np.float32)
        P = np.eye(3, dtype=np.float32)
        # print(kpts.dtype)
        kpts = kpts.reshape(-1, 1, 2)

        undistorted_kpts = cv2.fisheye.undistortPoints(kpts, K, D, R=R, P=P)
        undistorted_kpts = undistorted_kpts.reshape(-1, 2)
        undistorted_kpts = np.hstack(
            (undistorted_kpts, np.ones((undistorted_kpts.shape[0], 1)))
        )
        undistorted_kpts_list.append(undistorted_kpts)

    segment_id_pair = []
    for idx0, idx1 in preds['indices_pairs']:
        key = f'{idx0}_{idx1}'

        matches = all_matches[f'matches{idx0}_{key}'][0].cpu().numpy()
        confs = all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()

        seg_ids0 = kpt_seg_ids[idx0]
        seg_ids1 = kpt_seg_ids[idx1]

        valid_indices = np.flatnonzero(
            (matches >= 0) & (confs >= 0.02) & (seg_ids0 >= 0)
        )
        kpts0 = features['keypoints'][idx0].cpu().numpy()
        kpts1 = features['keypoints'][idx1].cpu().numpy()

        kpts0 = kpts0[valid_indices]
        seg_ids0 = seg_ids0[valid_indices]
        undists0 = undistorted_kpts_list[idx0][valid_indices]

        kpts1_idx = matches[valid_indices]
        kpts1 = kpts1[kpts1_idx]
        seg_ids1 = seg_ids1[kpts1_idx]
        undists1 = undistorted_kpts_list[idx1][kpts1_idx]

        valid_indices = np.flatnonzero(seg_ids1 >= 0)

        kpts0 = kpts0[valid_indices]
        seg_ids0 = seg_ids0[valid_indices]
        undists0 = undists0[valid_indices]

        kpts1 = kpts1[valid_indices]
        seg_ids1 = seg_ids1[valid_indices]
        undists1 = undists1[valid_indices]

        E = target_Ec1c0[key]

        undists0 = undists0[:, :, np.newaxis]
        temp = E @ undists0

        undists1 = undists1[:, np.newaxis, :]
        epipolar_constrains = (undists1 @ temp).flatten()

        # print(
        #     f"{idx0} - {idx1} shape : {epipolar_constrains.shape[0]} /\n mean: {epipolar_constrains}"
        # )
        logging.info(
            f"{timestamp_count}_{idx0} - {idx1} shape : {epipolar_constrains.shape[0]} / mean: {epipolar_constrains}"
        )

        threshold = epipolar_threshold[key]

        epipolar_constrains = np.abs(epipolar_constrains) < threshold

        before_epi0 = kpts0
        kpts0 = kpts0[epipolar_constrains]
        seg_ids0 = seg_ids0[epipolar_constrains]

        before_epi1 = kpts1
        kpts1 = kpts1[epipolar_constrains]
        seg_ids1 = seg_ids1[epipolar_constrains]

        unique_ids0 = np.unique(seg_ids0.flatten())
        unique_ids1 = np.unique(seg_ids1.flatten())

        id_idx0 = {}
        for idx, id in enumerate(unique_ids0):
            id_idx0[id] = idx

        id_idx1 = {}
        for idx, id in enumerate(unique_ids1):
            id_idx1[id] = idx

        rows = unique_ids0.shape[0]
        cols = unique_ids1.shape[0]

        hit_mat = np.zeros((rows, cols))

        for id0, id1 in zip(seg_ids0, seg_ids1):
            r = id_idx0[id0]
            c = id_idx1[id1]
            hit_mat[r, c] += 1

        for i in range(hit_mat.shape[0]):
            if np.sum(hit_mat[i] > 4) > 1:
                hit_mat[i] = 0

        for j in range(hit_mat.shape[1]):
            if np.sum(hit_mat[:, j] > 4) > 1:
                hit_mat[:, j] = 0

        matched_indices = np.array(solve_dense(-hit_mat)).T
        matches = []

        for m in matched_indices:
            if hit_mat[m[0], m[1]] > 4:
                matches.append(
                    (unique_ids0[m[0]], unique_ids1[m[1]], hit_mat[m[0], m[1]])
                )

        segment_id_pair += matches

        img0 = images[idx0].copy()
        img1 = images[idx1].copy()

        for i, (id0, id1, _) in enumerate(matches):
            seg0 = tracklet_db.query_segment_tracklet(id0)
            mask0 = seg0.query_segment_mask(cam_states[idx0].cam_id)
            img0 = draw_mask(img0, mask0, i, 0.8)
            seg1 = tracklet_db.query_segment_tracklet(id1)
            mask1 = seg1.query_segment_mask(cam_states[idx1].cam_id)
            img1 = draw_mask(img1, mask1, i, 0.8)

        stitched_img = np.concatenate([img0, img1], axis=1)
        vis_img = stitched_img.copy()

        vis_lines_before = np.concatenate(
            [before_epi0, before_epi1], axis=1
        ).astype(np.int32)
        vis_lines_before[:, 2] += img0.shape[1]

        vis_lines = np.concatenate([kpts0, kpts1], axis=1).astype(np.int32)
        vis_lines[:, 2] += img0.shape[1]

        for i, line in enumerate(vis_lines_before):
            c = (0, 0, 0)
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)

        for i, line in enumerate(vis_lines):
            c = colors[i % len(colors)]
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)
        cv2.putText(
            vis_img,
            f"{timestamp_count}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )
        match_video_frames[key].append(vis_img)

    segment_id_pairs_dict[timestamp] = segment_id_pair

    del torch_images, features, preds
    torch.cuda.empty_cache()

video_pose_whs = []
for cam_param in cam_params:
    video_pose_whs.append(cam_param.video_pos_wh)

timestamp_count = 0
video_tracklet_frames = []
for _, cam_states in cam_states_dict.items():
    timestamp_count += 1
    if timestamp_count < skip:
        continue

    if skip + dura != 0 and timestamp_count > skip + dura:
        break

    seg_frames = []
    tracklet_frames = []
    for cam_state in cam_states:
        tracklet_frame = cam_state.draw_tracklets()

        tracklet_frames.append(tracklet_frame)
    video_tracklet_video_frame = collate_images(tracklet_frames, video_pose_whs)
    cv2.putText(
        video_tracklet_video_frame,
        f"{timestamp_count}",
        (40, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        2,
        (0, 255, 255),
        3,
        cv2.LINE_AA,
    )
    video_tracklet_frames.append(video_tracklet_video_frame)

seg_video_dir = os.path.join(
    f'./cache/{data_base_name}', f"before_merge_tracklet.mp4"
)
video_writer = cv2.VideoWriter(
    seg_video_dir,
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (video_tracklet_frames[0].shape[1], video_tracklet_frames[0].shape[0]),
)

for video_frame in video_tracklet_frames:
    video_writer.write(video_frame)

video_writer.release()

for idx_pair, frames in match_video_frames.items():
    match_video_path = os.path.join(
        f'./cache/{data_base_name}', f"match_{idx_pair}.mp4"
    )
    video_writer = cv2.VideoWriter(
        match_video_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        20,
        (frames[0].shape[1], frames[0].shape[0]),
    )

    for frame in frames:
        video_writer.write(frame)

    video_writer.release()

del match_video_frames

my_dict = {}

my_dict[frozenset([1, 2])] = "Value for 1 and 2"


import networkx as nx

G = nx.Graph()
init_prob = 0.6


def accumulate(a, b):
    return a * b / (a * b + (1 - a) * (1 - b))


for _, edges in segment_id_pairs_dict.items():
    for id0, id1, hit in edges:
        if G.has_edge(id0, id1):
            new_prob = accumulate(G[id0][id1]['weight'], init_prob)
            G[id0][id1]['weight'] = min(new_prob, 0.99)
        else:
            # G.add_edge(id0, id1, weight=hit)
            G.add_edge(id0, id1, weight=init_prob)

from scipy.sparse import csr_matrix

adjacency = csr_matrix(nx.to_scipy_sparse_array(G, weight='weight'))
# print(adjacency.todense())

from sknetwork.clustering import Leiden

leiden = Leiden(resolution=0.8)
labels = leiden.fit_predict(adjacency)

connected_components = {}
for seg_id, group_id in zip(G.nodes, labels):
    connected_components.setdefault(group_id, []).append(seg_id)

from typing import List, Dict


def merge_seg_tracklet(trls: List[SegmentTracklet]):
    merged_trl = tracklet_db.create_new_segment_tracklet()
    for trl in trls:
        merged_trl.cam_id_to_segment_mask.update(trl.cam_id_to_segment_mask)
        tracklet_db.delete_segment_tracklet(trl.id)
        for cam_id, mask in trl.cam_id_to_segment_mask.items():
            cam_state = cam_states_dict[cam_id.timestamp][cam_id.index]
            del cam_state.seg_trls[trl.id]
            cam_state.seg_trls[merged_trl.id] = merged_trl

    merged_trl.merged = True
    return merged_trl


new_ids = []
for _, connected_component in connected_components.items():
    # print(connected_component)
    to_merge_trl = []

    for id in connected_component:
        to_merge_trl.append(tracklet_db.query_segment_tracklet(id))

    new_trl = merge_seg_tracklet(to_merge_trl)
    new_ids.append(new_trl.id)

from IPython.display import SVG
from sknetwork.visualization import svg_graph

image = svg_graph(adjacency, labels=labels)

with open(f"./cache/{data_base_name}/graph_image.svg", "w") as file:
    file.write(image)

timestamp_count = 0

video_seg_frames = []
video_tracklet_frames = []
video_merged_tracklet_frames = []
for _, cam_states in cam_states_dict.items():
    timestamp_count += 1
    if timestamp_count < skip:
        continue

    if skip + dura != 0 and timestamp_count > skip + dura:
        break
    seg_frames = []
    tracklet_frames = []
    merged_tracklet_frames = []
    for cam_state in cam_states:
        seg_frame = cam_state.draw_segments()
        seg_frames.append(seg_frame)
        tracklet_frame = cam_state.draw_tracklets()
        tracklet_frames.append(tracklet_frame)
        merged_tracklet_frame = cam_state.draw_merged_tracklets()
        merged_tracklet_frames.append(merged_tracklet_frame)
    video_seg_frame = collate_images(seg_frames, video_pose_whs)
    video_tracklet_video_frame = collate_images(tracklet_frames, video_pose_whs)
    video_merged_tracklet_video_frame = collate_images(
        merged_tracklet_frames, video_pose_whs
    )
    cv2.putText(
        video_seg_frame,
        f"{timestamp_count}",
        (40, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        2,
        (0, 255, 255),
        3,
        cv2.LINE_AA,
    )
    cv2.putText(
        video_tracklet_video_frame,
        f"{timestamp_count}",
        (40, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        2,
        (0, 255, 255),
        3,
        cv2.LINE_AA,
    )
    cv2.putText(
        video_merged_tracklet_video_frame,
        f"{timestamp_count}",
        (40, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        2,
        (0, 255, 255),
        3,
        cv2.LINE_AA,
    )
    video_seg_frames.append(video_seg_frame)
    video_tracklet_frames.append(video_tracklet_video_frame)
    video_merged_tracklet_frames.append(video_merged_tracklet_video_frame)

seg_video_dir = os.path.join(f'./cache/{data_base_name}', f"seg.mp4")
video_writer = cv2.VideoWriter(
    seg_video_dir,
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (video_seg_frames[0].shape[1], video_seg_frames[0].shape[0]),
)


for video_frame in video_seg_frames:
    video_writer.write(video_frame)

video_writer.release()
del video_seg_frames

tracklet_video_dir = os.path.join(f'./cache/{data_base_name}', f"tracklet.mp4")
video_writer = cv2.VideoWriter(
    tracklet_video_dir,
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (
        video_tracklet_frames[0].shape[1],
        video_tracklet_frames[0].shape[0],
    ),
)

for video_frame in video_tracklet_frames:
    video_writer.write(video_frame)

video_writer.release()
del video_tracklet_frames

merged_tracklet_video_dir = os.path.join(
    f'./cache/{data_base_name}', f"merged_tracklet.mp4"
)
video_writer = cv2.VideoWriter(
    merged_tracklet_video_dir,
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (
        video_merged_tracklet_frames[0].shape[1],
        video_merged_tracklet_frames[0].shape[0],
    ),
)

for video_frame in video_merged_tracklet_frames:
    video_writer.write(video_frame)

video_writer.release()
del video_merged_tracklet_frames

processing 10220121091342 1
processing 10220173828551 2
processing 10220210754944 3
processing 10220273221406 4
processing 10220318588609 5
processing 10220368535128 6
processing 10220417838161 7
processing 10220469495320 8
processing 10220527361825 9
processing 10220577511746 10
processing 10220629711330 11
processing 10220660157933 12
processing 10220719226478 13
processing 10220767216906 14
processing 10220820523326 15
processing 10220869306463 16
processing 10220919208532 17
processing 10220963780753 18
processing 10221020134323 19
processing 10221072150667 20
processing 10221124608248 21
processing 10221166214506 22
processing 10221225282571 23
processing 10221267973424 24
processing 10221321412522 25
processing 10221369096952 26
processing 10221420304778 27
processing 10221469482238 28
processing 10221516243713 29
processing 10221567528279 30
processing 10221621435335 31
processing 10221668447990 32
processing 10221725684545 33
processing 10221769253967 34
processing 102218171766